# Demo 12g — The transformer

**Class 12 · T3–T4** (slides: `slides/12_neural_networks.md`)

* **T3:** one encoder block (self-attention, feed-forward, residuals, layer norm): its parameters by formula and by Keras, and what it does on the memory task at a lag where the RNN, the LSTM and the GRU fail; the attention weights.
* **T4:** the order task with and without positions; room temperature forecasting; BBC news (1000 articles) with a small transformer against tf-idf and logistic regression, then a learning curve on 32 000 AG News articles: the transformer needs more data.

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")
import keras
import numpy as np
import polars as pl
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import train_test_split

from faa import data, demo, neural, plotlib
from faa.demo import show, slide

demo.setup("12")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0

## Slide: Parameters of a transformer block (T3)

Attention: $W_Q, W_K, W_V, W_O$ are $d \times d$ in total (heads split $d$, they do not add parameters), with biases $4d^2 + 4d$.
Feed-forward $d \to 4d \to d$: $8d^2 + 5d$. Two layer norms: $4d$. Total $12 d^2 + 13 d$.

In [ ]:
rows = []
for d in (32, 64, 128, 512):
    formula = 12 * d * d + 13 * d
    if d <= 128:
        inp = keras.layers.Input((10, d))
        a = keras.layers.MultiHeadAttention(4, d // 4)(inp, inp)
        x = keras.layers.LayerNormalization()(inp + a)
        f = keras.layers.Dense(d)(keras.layers.Dense(4 * d, activation="relu")(x))
        out = keras.layers.LayerNormalization()(x + f)
        counted = neural.n_params(keras.Model(inp, out))
    else:
        counted = None
    rows.append({"d": d, "formula 12 d^2 + 13 d": formula, "Keras": counted})
slide("Parameters of a transformer block")
show(pl.DataFrame(rows))
print("one LSTM layer with u = d inputs and units: 8 d^2 + 4 d =", [8 * d * d + 4 * d for d in (32, 64, 128, 512)])
print(f"d = 512: {(12 * 512 * 512 + 13 * 512) / 1e6:.2f} million weights per block")
f, ax = plotlib.fig("col", 1.6)
ds = np.array([32, 64, 128])
ax.bar(np.arange(3) - 0.2, [4 * d * d + 4 * d for d in ds], 0.4, color=BLUE, label="attention")
ax.bar(np.arange(3) + 0.2, [8 * d * d + 5 * d for d in ds], 0.4, color=ORANGE, label="feed-forward")
ax.set_xticks(range(3), [f"$d$ = {d}" for d in ds])
ax.set_ylabel("weights")
ax.legend(frameon=False, fontsize=6)
plotlib.save(f, "t3_block_params")

## Slide: The memory task at lag 100 (T3)

Same task as Demo 12e: the first step holds the symbol, 100 steps of noise follow, the model answers from the last step. The RNN, the LSTM and the GRU stay at chance.
The encoder block (`d_model` 32, 2 heads, positions, 1 block) looks at step 0 directly.

In [ ]:
lag = 100
xs, ys = neural.memory_task(3000, lag, seed=1)
xt, yt = neural.memory_task(1000, lag, seed=2)
rows = []
for s in range(3):
    m = neural.transformer_classifier(xs.shape[1:], seed=s)
    m.fit(xs, ys, epochs=15, batch_size=64, verbose=0)
    rows.append(m.evaluate(xt, yt, verbose=0)[1])
slide("Transformer on the memory task at lag 100")
print(
    f"transformer accuracy over 3 seeds: {np.mean(rows):.3f} +- {np.std(rows):.3f} (chance 0.25); parameters {neural.n_params(m)}"
)
rec = {}
for kind in ("rnn", "lstm", "gru"):
    mr = neural.recurrent_model(kind, xs.shape[1:], clipnorm=1.0, seed=0)
    mr.fit(xs, ys, epochs=15, batch_size=64, verbose=0)
    rec[kind] = mr.evaluate(xt, yt, verbose=0)[1]
print("same data, recurrent layers (32 units, seed 0): " + ", ".join(f"{k.upper()} {v:.3f}" for k, v in rec.items()))
f, ax = plotlib.fig("col", 1.5)
names = ["RNN", "LSTM", "GRU", "transformer"]
vals = [rec["rnn"], rec["lstm"], rec["gru"], float(np.mean(rows))]
ax.bar(names, vals, color=[RED, BLUE, GREEN, ORANGE])
ax.axhline(0.25, color=GREY, ls=":", lw=0.8)
for k, v in enumerate(vals):
    ax.text(k, v + 0.03, f"{v:.2f}", ha="center", fontsize=6)
ax.set_ylim(0, 1.15)
ax.set_ylabel("test accuracy, lag 100")
plotlib.save(f, "t3_lag100")

## Slide: Where the last step looks (T3)

Attention weights of the last position (the one that answers) over the 101 positions, averaged over the heads and 200 test sequences, for lag 20.

In [ ]:
lag = 20
xs, ys = neural.memory_task(3000, lag, seed=1)
xt, yt = neural.memory_task(200, lag, seed=2)
m, wm = neural.transformer_classifier(xs.shape[1:], return_weights=True, seed=0)
m.fit(xs, ys, epochs=15, batch_size=64, verbose=0)
w = np.array(wm(xt))  # (batch, heads, n, n)
last = w[:, :, -1, :].mean((0, 1))
slide("Where the last step looks")
print(f"test accuracy at lag {lag}: {m.evaluate(xt, yt, verbose=0)[1]:.3f}")
print(
    f"attention of the last position: on step 0 {last[0]:.3f}, on each other step {last[1:].mean():.4f} (uniform would be {1 / (lag + 1):.4f})"
)
f, ax = plotlib.fig("col", 1.6)
ax.bar(np.arange(lag + 1), last, color=BLUE)
ax.set_xlabel("position $j$")
ax.set_ylabel("weight of the last query")
plotlib.save(f, "t3_last_query")

## Slide: The order task, with and without positions (T4)

One step carries marker A, one marker B; the label says which comes first. Attention without positions sees the same set of tokens in both cases.

In [ ]:
rows = []


def run_length(length):
    xs, ys = neural.order_task(4000, length, seed=1)
    xt, yt = neural.order_task(1000, length, seed=2)
    res = {}
    for name, kw in {"with positions": dict(positions=True), "no positions": dict(positions=False)}.items():
        a = []
        for s in range(3):
            m = neural.transformer_classifier(xs.shape[1:], n_out=2, readout="mean", seed=s, **kw)
            m.fit(xs, ys, epochs=25, batch_size=64, verbose=0)
            a.append(m.evaluate(xt, yt, verbose=0)[1])
        res[name] = np.mean(a)
    a = []
    for s in range(3):
        m = neural.recurrent_model("lstm", xs.shape[1:], n_out=2, seed=s, clipnorm=1.0)
        m.fit(xs, ys, epochs=25, batch_size=64, verbose=0)
        a.append(m.evaluate(xt, yt, verbose=0)[1])
    rows.append({"length": length, **res, "LSTM": np.mean(a)})


run_length(8)

In [ ]:
run_length(16)

In [ ]:
slide("Order task")
show(pl.DataFrame(rows))
f, ax = plotlib.fig("col", 1.5)
w = 0.25
for k, (key, c) in enumerate((("with positions", BLUE), ("no positions", RED), ("LSTM", GREEN))):
    ax.bar(
        np.arange(2) + (k - 1) * w,
        [r[key] for r in rows],
        w,
        color=c,
        label=key.replace("with positions", "positions").replace("no positions", "none"),
    )
ax.set_xticks(range(2), [f"length {r['length']}" for r in rows])
ax.axhline(0.5, color=GREY, ls=":", lw=0.8)
ax.set_ylabel("test accuracy")
ax.legend(
    frameon=False,
    fontsize=5.5,
    loc="lower center",
    ncol=3,
    bbox_to_anchor=(0.5, 1.0),
    columnspacing=0.8,
    handlelength=1.2,
)
plotlib.save(f, "t4_order")

## Slide: Forecasting with attention (T4)

Same data and protocol as Demo 12e (six hours ahead, last 15 % of the hours as test). One block, 32 units, 2 heads, three seeds.

In [ ]:
HOR = 6
Xtr, ytr, Xva, yva, Xte, yte, mu, sd = neural.climate_windows(24, HOR, 5)
dtr, dva, dte = ytr - Xtr[:, -1, 0], yva - Xva[:, -1, 0], yte - Xte[:, -1, 0]
scale = dtr.std()


def mae(p):
    return float(np.abs(np.ravel(p) - dte).mean() * sd)


maes = []
for s in range(3):
    m = neural.transformer_regressor(Xtr.shape[1:], seed=s)
    m.fit(
        Xtr,
        dtr / scale,
        epochs=60,
        batch_size=64,
        validation_data=(Xva, dva / scale),
        verbose=0,
        callbacks=[keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)],
    )
    maes.append(mae(m.predict(Xte, verbose=0).ravel() * scale))
slide("Forecasting with attention")
rows = [
    {"model": "persistence", "MAE (C)": mae(0 * dte), "std over seeds": 0.0},
    {
        "model": "ridge on the window",
        "MAE (C)": mae(Ridge(1.0).fit(Xtr.reshape(len(Xtr), -1), dtr).predict(Xte.reshape(len(Xte), -1))),
        "std over seeds": 0.0,
    },
    {"model": "transformer block", "MAE (C)": float(np.mean(maes)), "std over seeds": float(np.std(maes))},
]
show(pl.DataFrame(rows))
f, ax = plotlib.fig("col", 1.2)
ax.barh(
    [r["model"].replace(" on the window", "").replace(" block", "") for r in rows][::-1],
    [r["MAE (C)"] for r in rows][::-1],
    xerr=[r["std over seeds"] for r in rows][::-1],
    color=[BLUE, GREY, GREY],
)
ax.set_xlabel("test MAE (°C)")
plotlib.save(f, "t4_forecast")

## Slide: A transformer on text (T4)

1 000 BBC articles, five categories, 60 % train, 20 % validation, 20 % test. The transformer reads the first 200 words (vocabulary of the 3 000 most frequent training words, padding and unknown as ids 0 and 1), one block, 32 units, trained from scratch.
The baseline of Class 09: tf-idf and logistic regression.

In [ ]:
texts, cats = data.load_news()
labels = {c: i for i, c in enumerate(sorted(set(cats)))}
yv = np.array([labels[c] for c in cats])
tx_tr, tx_te, y_tr, y_te = train_test_split(texts, yv, test_size=0.4, random_state=SEED, stratify=yv)
tx_va, tx_te, y_va, y_te = train_test_split(tx_te, y_te, test_size=0.5, random_state=SEED, stratify=y_te)
cv = CountVectorizer(max_features=3000).fit(tx_tr)
analyzer = cv.build_analyzer()
index = {w: i + 2 for i, w in enumerate(cv.get_feature_names_out())}  # 0 = padding, 1 = unknown word


def ids(texts_, length=200):
    out = np.zeros((len(texts_), length), dtype="int32")
    for r, t in enumerate(texts_):
        tok = [index.get(w, 1) for w in analyzer(t)][:length]
        out[r, : len(tok)] = tok
    return out


tfidf = TfidfVectorizer(max_features=3000, stop_words="english").fit(tx_tr)
lr = LogisticRegression(max_iter=2000).fit(tfidf.transform(tx_tr), y_tr)
accs = []
for s in range(3):
    m = neural.text_transformer(3002, 200, seed=s)
    m.fit(
        ids(tx_tr),
        y_tr,
        epochs=30,
        batch_size=32,
        validation_data=(ids(tx_va), y_va),
        verbose=0,
        callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
    )
    accs.append(m.evaluate(ids(tx_te), y_te, verbose=0)[1])
slide("A transformer on text")
print(f"tf-idf + logistic regression: test accuracy {lr.score(tfidf.transform(tx_te), y_te):.3f}")
print(
    f"transformer from scratch ({neural.n_params(m)} parameters): {np.mean(accs):.3f} +- {np.std(accs):.3f} over 3 seeds"
)
f, ax = plotlib.fig("col", 1.3)
ax.barh(
    ["transformer", "tf-idf + LR"],
    [np.mean(accs), lr.score(tfidf.transform(tx_te), y_te)],
    xerr=[np.std(accs), 0],
    color=[BLUE, GREY],
)
ax.set_xlim(0.8, 1.0)
ax.set_xlabel("test accuracy")
plotlib.save(f, "t4_text")

## Slide: The transformer needs more data (T4)

1000 articles are too few for a transformer trained from scratch.
AG News: 32 000 training articles (headline and first sentence, 4 categories: world, sports, business, scitech) and 4 000 test articles.
For each training size $n$ the same two models are trained on the first $n$ articles (shuffled), 3 seeds for the transformer; the last 10 % of the $n$ articles are its validation part.
The vocabulary (5 000 words) is read from the training texts; each article is cut to 48 tokens.

In [ ]:
trx, try_, tex, tey = data.load_ag_news()
cats = sorted(set(try_))
ytr_a = np.array([cats.index(c) for c in try_])
yte_a = np.array([cats.index(c) for c in tey])
perm = np.random.default_rng(0).permutation(len(trx))
trx, ytr_a = [trx[i] for i in perm], ytr_a[perm]
cv_a = CountVectorizer(max_features=5000).fit(trx)
an_a = cv_a.build_analyzer()
index_a = {w: i + 2 for i, w in enumerate(cv_a.get_feature_names_out())}


def ids_a(texts_, length=48):
    out = np.zeros((len(texts_), length), dtype="int32")
    for r, t in enumerate(texts_):
        tok = [index_a.get(w, 1) for w in an_a(t)][:length]
        out[r, : len(tok)] = tok
    return out


Itr_a, Ite_a = ids_a(trx), ids_a(tex)
curve = {}


def run_size(n):
    tf = TfidfVectorizer(max_features=5000, stop_words="english").fit(trx[:n])
    lr_a = LogisticRegression(max_iter=2000).fit(tf.transform(trx[:n]), ytr_a[:n])
    nv = max(n // 10, 50)
    accs_n = []
    for s in range(3):
        m = neural.text_transformer(5002, 48, n_out=4, seed=s)
        m.fit(
            Itr_a[: n - nv],
            ytr_a[: n - nv],
            epochs=30,
            batch_size=64,
            validation_data=(Itr_a[n - nv : n], ytr_a[n - nv : n]),
            verbose=0,
            callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)],
        )
        accs_n.append(m.evaluate(Ite_a, yte_a, verbose=0)[1])
    curve[n] = (lr_a.score(tf.transform(tex), yte_a), float(np.mean(accs_n)), float(np.std(accs_n)))
    print(f"n = {n:5d}: tf-idf + LR {curve[n][0]:.3f}   transformer {curve[n][1]:.3f} +- {curve[n][2]:.3f}")


for n in (250, 500, 1000, 2000):
    run_size(n)

In [ ]:
for n in (4000, 8000):
    run_size(n)

In [ ]:
for n in (16000, 32000):
    run_size(n)

In [ ]:
slide("More data for the transformer")
print(f"{len(trx)} training articles, {len(tex)} test articles, {len(cats)} categories")
gap = {n: curve[n][0] - curve[n][1] for n in curve}
print("gap (tf-idf minus transformer): " + ", ".join(f"{n}: {g:+.3f}" for n, g in gap.items()))
f, ax = plotlib.fig("col", 1.8)
ns = list(curve)
ax.errorbar(
    ns,
    [curve[n][1] for n in ns],
    yerr=[curve[n][2] for n in ns],
    color=BLUE,
    marker="o",
    ms=3,
    lw=1,
    label="transformer",
)
ax.plot(ns, [curve[n][0] for n in ns], color=GREY, marker="s", ms=3, lw=1, label="tf-idf + LR")
ax.set_xscale("log")
ax.set_xlabel("training articles $n$")
ax.set_ylabel("test accuracy")
ax.legend(frameon=False, fontsize=6)
plotlib.save(f, "t4_more_data")